# Fazer
## Imediatos

[ ][ ] Calcular aprovados apenas pela média_final, sem exame_especial

[ ][ ] Gerar gráficos de status final do aluno com todas as turmas

[x][ ] statusEntrega: mudar label e colocar qtd de alunos

[x][ ] entregas por turma: fazer gráfico


[ ][ ] fazer gráfico relacionando frequência e nota final


## Planejar
[ ] Definir limiar de faltas para remoção

# Preparação do ambiente colab

In [ ]:
!pip install plotly==6.1.1
!pip install kaleido
!plotly_get_chrome -y



   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.1/16.1 MB 99.7 MB/s eta 0:00:00
  Attempting uninstall: plotly
    Found existing installation: plotly 5.24.1
    Uninstalling plotly-5.24.1:
      Successfully uninstalled plotly-5.24.1
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.0/69.0 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.3/49.3 kB 3.6 MB/s eta 0:00:00
Installing Chrome for Plotly...
Chrome installed successfully.
The Chrome executable is now located at: /usr/local/lib/python3.12/dist-packages/choreographer/cli/browser_exe/chrome-linux64/chrome


In [ ]:
# dependencies for kaleido
!sudo apt update && sudo apt-get install libnss3 libatk-bridge2.0-0 libcups2 libxcomposite1 libxdamage1 libxfixes3 libxrandr2 libgbm1 libxkbcommon0 libpango-1.0-0 libcairo2 libasound2


Hit:1 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:2 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:3 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Hit:4 https://cli.github.com/packages stable InRelease
Get:5 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Get:6 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Get:7 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:8 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:9 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Hit:10 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Hit:11 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Get:12 http://archive.ubuntu.com/ubuntu jammy-updates/universe amd64 Packages [1,595 kB]
Get:13 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x8

In [ ]:
import pandas as pd
import numpy as np
import os
import matplotlib as mpl
mpl.use('Agg')
import matplotlib.pyplot as plt
import kaleido
import plotly.express as px
import plotly.graph_objects as go

In [ ]:
# Criação de diretórios para salvar gráficos
os.mkdir("tabelasTotais")
os.mkdir("boxplots")
os.mkdir("relacaoNotasEntregas")
os.mkdir("entregasPorQuestao")
os.mkdir("entregasPorQuestao_correction")
os.mkdir("qtd_entregas")
os.mkdir('statusFinal')
os.mkdir('statusEntregas')
os.mkdir('boxPlot_notas')
os.mkdir('entregasPorQuestao/geral')
os.mkdir('entregasPorQuestao/grade')
os.mkdir('entregasPorQuestao/grade_accounting')
os.mkdir('entregas_notas_SSR')

In [ ]:

df_geral = pd.read_csv("full_data_merged_classesRevised_comReprFaltas.csv", sep=";")


In [ ]:
print("Number of rows in df_geral:", len(df_geral))
print("Number of deliveries with missing question_name:", df_geral['question_name'].isnull().sum())

Number of rows in df_geral: 21761
Number of deliveries with missing question_name: 0


In [ ]:
df_entregas = df_geral.groupby(["user_id", "class_name", "task_name"]).size().to_frame("qtd_entregas")

In [ ]:
df_entregas

qtd_entregas
user_id                          class_name   task_name                                                       
01eb90c1bea98bf99e28c62db1f3807c 9_10 (24.2)  [AP-2_3] Variáveis, Expressões, Entrada e Saída                9
                                              [AP-4.1] Estrutura de Decisão Simples                         10
                                              [AP-4.2] Estruturas de Decisão Aninhadas                      12
                                              [AP-4.3] Estrutura de Decisão - Atividade Compl...             6
                                              [AP-5.1] Estrutura de Repetição WHILE                         14
...                                                                                                        ...
fe4412c0ea344fa8fd957074fdeac75a 11_12 (24.2) [AP-7.1] Estrutura Homogênea: VETOR                            6
                                              [AP-7.2] Estrutura Homogênea: VETOR                            3
                                              [AP-7.4] Estrutura Homogênea: MATRIZ                           3
                                              [AP-S1] Simulado da Prova Teórica Unificada 1                  2
                                              [AP-S2] Simulado da Prova Teórica Unificada 2                  4

[3855 rows x 1 columns]

In [ ]:
df_entregas.reset_index(inplace=True)

In [ ]:
# Filter out rows where 'media_final' is null
df_not_dropped = df_geral.dropna(subset=['media_final'])
df_not_dropped = df_geral.dropna(subset=['faltas'])

# Count the number of unique user IDs in the filtered DataFrame
num_students_not_dropped = df_not_dropped['user_id'].nunique()

print(f"Number of students with a final grade (did not drop the class): {num_students_not_dropped}")

Number of students with a final grade (did not drop the class): 370


In [ ]:
# # @title
# from google.colab import sheets
# sheet = sheets.InteractiveSheet(df=df_entregas)

In [ ]:
# define detalhes extra dos nomes dos arquivos
detalhamento = "_(comReprFaltas)"

# Quantidade de Entregas
### Quantidade de entregas por aluno, separado por turma

In [ ]:
def totalEntregas_bar(df_entregas):
    total_deliveries_per_user_and_class = df_entregas.groupby(["class_name", "user_id"])["qtd_entregas"].sum()

    total_deliveries_per_user_and_class.sort_values(ascending=True, inplace=True)

    for class_name, data in total_deliveries_per_user_and_class.groupby("class_name"):
        plt.figure(figsize=(12, 6))
        ax = data.plot(kind="bar")
        plt.title(f"Total de entregas feitas por cada usuário na turma {class_name}")
        plt.xlabel("User ID")
        plt.ylabel("Número de entregas")
        plt.xticks([]) # Hide x-axis labels for readability if there are many users

        # Add total deliveries on top of each bar
        for p in ax.patches:
            ax.annotate(f'{p.get_height():.0f}', (p.get_x() + p.get_width() / 2., p.get_height()),
                        ha='center', va='center', xytext=(0, 5), textcoords='offset points')

        # Add horizontal lines
        plt.axhline(y=50, color='r', linestyle='--', label='50 entregas')
        plt.axhline(y=100, color='g', linestyle='--', label='100 entregas')
        plt.axhline(y=150, color='b', linestyle='--', label='150 entregas')
        plt.legend()

        plt.tight_layout()
        plt.show()

        # Save the figure
        plt.savefig(f"tabelasTotais/TotalEntregasUsuario_Turma_{class_name.replace(' ', '_')}.png")
        plt.close() # Close the figure to avoid displaying it twice

totalEntregas_bar(df_entregas)

In [ ]:
def plot_totalEntregas_box(df_entregas):
    total_deliveries_per_user_and_class = df_entregas.groupby(["class_name", "user_id"])["qtd_entregas"].sum()

    total_deliveries_per_user_and_class.sort_values(ascending=True, inplace=True)

    fig = px.box(
        df_entregas,
        x="class_name",
        y="qtd_entregas",
        title=f"Distribuição de entregas feitas por cada usuário na turma {detalhamento}",
        points="outliers"
    )
    fig.update_layout(xaxis_title="Turma", yaxis_title="Número de entregas")
    fig.write_image(f"qtd_entregas/boxplot_entregas_turma{detalhamento}.png")
    fig.write_html(f"qtd_entregas/html_boxplot_entregas_turma{detalhamento}.html")
    fig.show()


def sheet_totalEntregas_box(df_entregas):
    total_deliveries_per_user_and_class = df_entregas.groupby(["class_name", "user_id"])["qtd_entregas"].sum()

    total_deliveries_per_user_and_class.sort_values(ascending=True, inplace=True)

    class_names = []
    q1_values = []
    median_values = []
    q3_values = []
    average_values = []
    max_values = []
    min_values = []

    for class_name, data in total_deliveries_per_user_and_class.groupby("class_name"):
        q1 = np.percentile(data, 25)
        median = np.percentile(data, 50)
        q3 = np.percentile(data, 75)
        average = np.mean(data)
        max_val = np.max(data)
        min_val = np.min(data)

        class_names.append(class_name)
        q1_values.append(q1)
        median_values.append(median)
        q3_values.append(q3)
        average_values.append(average)
        max_values.append(max_val)
        min_values.append(min_val)


    average_values = [f"{average:.2f}" for average in average_values]
    max_values = [f"{max_val:.2f}" for max_val in max_values]
    min_values = [f"{min_val:.2f}" for min_val in min_values]
    class_names = [f"<b>{name}</b>" for name in class_names]


    fig = go.Figure(data=[go.Table(header=dict(values=['Turma', 'Média', 'Mediana', 'Máximo', 'Mínimo', 'Q1', 'Q3']),
                                    cells=dict(values=[class_names, average_values, median_values, max_values, min_values, q1_values, q3_values],
                                               fill=dict(color=["paleturquoise", "lavender"]),
                                               align=["right", "center"]
                                               ))

                            ])

    fig.show()
    fig.write_image(f"qtd_entregas/tabelaAnalise_entregas_turma{detalhamento}.png")



plot_totalEntregas_box(df_entregas)
sheet_totalEntregas_box(df_entregas)

In [ ]:
# !zip -r qtdEntregas.zip qtd_entregas

# Análise de distribuição de notas de cada classe (boxplots)


In [ ]:
def mesclaNotasExameFinal(df):
    # Use np.maximum to compare 'media_final' and 'exame_especial' element-wise
    # np.maximum treats NaN as smaller than any number, which fits the requirement
    df_alterado = df.copy()
    df_alterado["exame_especial"] = df["exame_especial"].fillna(0)
    df_alterado['media_final'] = np.maximum(df_alterado['media_final'], df_alterado['exame_especial'])
    return df_alterado

In [ ]:
def mesclaNotasExameFinal(df):
    # Use np.maximum to compare 'media_final' and 'exame_especial' element-wise
    # np.maximum treats NaN as smaller than any number, which fits the requirement
    df_alterado = df.copy()
    df_alterado["exame_especial"] = df["exame_especial"].fillna(0)
    df_alterado['media_final'] = np.maximum(df_alterado['media_final'], df_alterado['exame_especial'])
    return df_alterado


def plot_notas_box(df, escopo, max_absences=None):
    # escopo se refere a contexto sendo avaliado: sem exame final, ou com exame final

    if escopo == "exame_especial":
        df = mesclaNotasExameFinal(df)
    df_cleaned = df.dropna(subset=["media_final"])

    if max_absences is not None:
        df_cleaned = df_cleaned[df_cleaned['faltas'] <= max_absences].copy()

    nome_escopo = "Sem Exame Final" if escopo == "media_final" else "Com Exame Final"
    fig = px.box(
        df_cleaned,
        x="class_name",
        y="media_final",
        title=f"Notas {nome_escopo} - Max Absences: {max_absences}",
        points="outliers"
    )
    fig.update_layout(xaxis_title="Turma", yaxis_title="Nota")
    fig.write_image(f"boxPlot_notas/boxplot_notas_turma{detalhamento}_maxAbsences{max_absences}.png")
    fig.write_html(f"boxPlot_notas/boxplot_notas_turma{detalhamento}_maxAbsences{max_absences}.html")
    fig.show()

def sheet_notas_box(df, escopo, max_absences=None):

    if escopo == "exame_especial":
        df = mesclaNotasExameFinal(df)

    df_numeric = df.dropna(subset=["media_final"]) # Drop rows where the target column is NaN

    if max_absences is not None:
        df_numeric = df_numeric[df_numeric['faltas'] <= max_absences].copy()

    nome_escopo = "Sem Exame Final" if escopo == "media_final" else "Com Exame Final"

    class_names = []
    q1_values = []
    median_values = []
    q3_values = []
    average_values = []
    max_values = []
    min_values = []
    lista_num_aprovados = []
    lista_num_reprovados = []
    lista_total_alunos = [] # New list for total students

    df_by_student = df_numeric.drop_duplicates(subset=['user_id'])

    # Process each class
    for class_name, data in df_by_student.groupby("class_name"):
        num_aprovados = data[data['obs'] == "APROV"].shape[0]
        num_reprovados = data[data['obs'] == 'REPRV'].shape[0]
        total_students_class = data['user_id'].nunique()
        numeric_data = data["media_final"]
        q1 = np.percentile(numeric_data, 25)
        median = np.percentile(numeric_data, 50)
        q3 = np.percentile(numeric_data, 75)
        average = np.mean(numeric_data)
        max_val = np.max(numeric_data)
        min_val = np.min(numeric_data)

        lista_num_aprovados.append(num_aprovados)
        lista_num_reprovados.append(num_reprovados)
        lista_total_alunos.append(total_students_class) # Append for each class
        class_names.append(class_name)
        q1_values.append(q1)
        median_values.append(median)
        q3_values.append(q3)
        average_values.append(average)
        max_values.append(max_val)
        min_values.append(min_val)

    # Process all classes combined
    total_students_global = df_by_student['user_id'].nunique()
    aprovados_global = df_by_student[df_by_student['obs'] == "APROV"].shape[0]
    reprovados_global = df_by_student[df_by_student['obs'] == 'REPRV'].shape[0]
    numeric_data_global = df_by_student["media_final"]
    q1_global = np.percentile(numeric_data_global, 25)
    median_global = np.percentile(numeric_data_global, 50)
    q3_global = np.percentile(numeric_data_global, 75)
    average_global = np.mean(numeric_data_global)
    max_val_global = np.max(numeric_data_global)
    min_val_global = np.min(numeric_data_global)

    lista_num_aprovados.append(aprovados_global)
    lista_num_reprovados.append(reprovados_global)
    lista_total_alunos.append(total_students_global) # Append global total
    class_names.append("<b>Geral</b>")
    q1_values.append(q1_global)
    median_values.append(median_global)
    q3_values.append(q3_global)
    average_values.append(average_global)
    max_values.append(max_val_global)
    min_values.append(min_val_global)

    average_values = [f"{average:.2f}" for average in average_values]
    max_values = [f"{max_val:.2f}" for max_val in max_values]
    min_values = [f"{min_val:.2f}" for min_val in min_values]
    class_names = [f"<b>{name}</b>" for name in class_names]
    q1_values = [f"{q1:.2f}" for q1 in q1_values]
    q3_values = [f"{q3:.2f}" for q3 in q3_values]

    fig = go.Figure(data=[go.Table(header=dict(values=['Turma', 'Total Alunos', 'Aprovados', 'Reprovados','Média', 'Mediana', 'Máximo', 'Mínimo', 'Q1', 'Q3']),
                                    cells=dict(values=[class_names, lista_total_alunos, lista_num_aprovados, lista_num_reprovados, average_values, median_values, max_values, min_values, q1_values, q3_values],
                                               fill=dict(color=["paleturquoise", "lavender"]),
                                               align=["right", "center"]
                                               ))

                            ])
    fig.update_layout(
    title=dict(
        text=f"Nota {nome_escopo} - Max Absences: {max_absences}"
    ))
    fig.write_image(f"boxPlot_notas/tabelaAnalise_notas_turma{detalhamento}_maxAbsences{max_absences}.png")
    fig.show()


plot_notas_box(df_geral, "exame_especial", max_absences=100)
plot_notas_box(df_geral, "exame_especial", max_absences=35)

sheet_notas_box(df_geral, "exame_especial", max_absences=100)
sheet_notas_box(df_geral, "exame_especial", max_absences=35)

In [ ]:
!zip -r boxPlot_notas.zip boxPlot_notas

  adding: boxPlot_notas/ (stored 0%)
  adding: boxPlot_notas/boxplot_notas_turma_(comReprFaltas)_maxAbsences100.png (deflated 16%)
  adding: boxPlot_notas/tabelaAnalise_notas_turma_(comReprFaltas)_maxAbsences35.png (deflated 16%)
  adding: boxPlot_notas/boxplot_notas_turma_(comReprFaltas)_maxAbsences35.png (deflated 15%)
  adding: boxPlot_notas/boxplot_notas_turma_(comReprFaltas)_maxAbsences100.html (deflated 72%)
  adding: boxPlot_notas/tabelaAnalise_notas_turma_(comReprFaltas)_maxAbsences100.png (deflated 17%)
  adding: boxPlot_notas/boxplot_notas_turma_(comReprFaltas)_maxAbsences35.html (deflated 72%)


# Relação entre qtd de entregas e nota (geral)

In [ ]:
# Removing duplicates, keeping the first entry for each user and final grade
# criando dataframe com user_id e media_final, para análises com a nota

notas_alunos = df_geral.drop_duplicates(subset=['user_id', 'media_final'])

In [ ]:
# # @title
# def plot_entregasNotas_scatter(df, escopo):
#     total_deliveries_per_user_and_class = df.groupby(["class_name", "user_id"])["qtd_entregas"].sum()

#     total_deliveries_per_user_and_class.sort_values(ascending=True, inplace=True)
#     # Merge total deliveries per user and class with the final grades
#     merged_deliveries_grades = total_deliveries_per_user_and_class.reset_index().merge(
#         notas_alunos[["user_id", "media_final"]], on="user_id", how="left"
#     )

#     # Create a new column to categorize grades
#     merged_deliveries_grades['grade_category'] = merged_deliveries_grades['media_final'].apply(
#         lambda x: 'Média Final >= 6' if x >= 6 else 'Média Final < 6'
#     )

#     # Calculate the average of total deliveries
#     average_deliveries = merged_deliveries_grades['qtd_entregas'].mean()

#     # Plot the relationship between total deliveries and final grades using Plotly Express
#     fig = px.scatter(
#         merged_deliveries_grades,

#         x='qtd_entregas',
#         y='media_final',
#         color='grade_category',
#         title='Relação entre total de entregas e nota final',
#         labels={'qtd_entregas': 'Total de entregas', 'media_final': 'Média Final'},
#         color_discrete_map={'Média Final < 6': 'red', 'Média Final >= 6': 'blue'},
#         # trendline="ols"
#     )

#     fig.add_hline(y=6, line_dash="dash", line_color="black", annotation_text="Média Final >= 6")

#     aprovados = merged_deliveries_grades[merged_deliveries_grades['grade_category'] == 'Média Final >= 6'].shape[0]
#     aprovados_empenhados = merged_deliveries_grades[(merged_deliveries_grades['grade_category'] == 'Média Final >= 6') & (merged_deliveries_grades["qtd_entregas"] > average_deliveries)].shape[0]
#     total_alunos = merged_deliveries_grades["user_id"].nunique()

#     # Add a vertical line at the average number of deliveries
#     fig.add_vline(x=average_deliveries, line_dash="dash", line_color="black", annotation_text=f"Average Deliveries: {average_deliveries:.2f}")

#     fig.add_annotation(text=f"Total de alunos: <b>{total_alunos}</b>",
#                             align="left",
#                             showarrow=False,
#                             xref="paper",
#                             yref="paper",
#                             x=1.02,  # Adjust x and y to position it near your legend
#                             y=0.8,
#                             xanchor="left",
#                             font=dict(size=12, color="black"),
#                             bgcolor="rgba(255,255,255,1)"
#                             )
#     # total de aprovados
#     fig.add_annotation(text=f"Total aprovados: {aprovados}",
#                             align="left",
#                             showarrow=False,
#                             xref="paper",
#                             yref="paper",
#                             x=1.02,  # Adjust x and y to position it near your legend
#                             y=0.7,
#                             xanchor="left",
#                             font=dict(size=12, color="black"),
#                             bgcolor="rgba(255,255,255,1)"
#                             )
#     # total de aprovados que entregaram mais que a média de atividades
#     fig.add_annotation(text=f"Aprovados<br>Empenhados: {aprovados_empenhados}",
#                             align="left",
#                             showarrow=False,
#                             xref="paper",
#                             yref="paper",
#                             x=1.02,  # Adjust x and y to position it near your legend
#                             y=0.55,
#                             xanchor="left",
#                             font=dict(size=12, color="black"),
#                             bgcolor="rgba(255,255,255,1)"
#                             )
#     # porcentagem de aprovados que entregaram mais que a média de atividades

#     fig.add_annotation(text=f"% empenhados<br> (<b>em aprovados</b>): %{(aprovados_empenhados/aprovados)*100:.2f}",
#                             align="left",
#                             showarrow=False,
#                             xref="paper",
#                             yref="paper",
#                             x=1.02,  # Adjust x and y to position it near your legend
#                             y=0.4,
#                             xanchor="left",
#                             font=dict(size=12, color="black"),
#                             bgcolor="rgba(255,255,255,1)"
#                             )

#     fig.add_annotation(text=f"% empenhados<br> (<b>no total</b>): %{(aprovados_empenhados/total_alunos)*100:.2f}",
#                             align="left",
#                             showarrow=False,
#                             xref="paper",
#                             yref="paper",
#                             x=1.02,  # Adjust x and y to position it near your legend
#                             y=0.25,
#                             xanchor="left",
#                             font=dict(size=12, color="black"),
#                             bgcolor="rgba(255,255,255,1)"
#                             )

#     fig.show()

#     # Save the plot as an HTML file
#     fig.write_image("relacaoNotasEntregas/Relacao_EntregasNota.png")

# plot_entregasNotas_scatter(df_entregas, "media_final")

# Relação entre qtd de entregas e nota (por turma) - DEPRECATED

In [ ]:
# # @title
# def plot_entregasNotasTurma_scatter(df, escopo):
#     total_deliveries_per_user_and_class = df.groupby(["class_name", "user_id"])["qtd_entregas"].sum()

#     total_deliveries_per_user_and_class.sort_values(ascending=True, inplace=True)

#      # Merge total deliveries per user and class with the final grades
#     merged_deliveries_grades = total_deliveries_per_user_and_class.reset_index().merge(
#         notas_alunos[["user_id", escopo]], on="user_id", how="left"
#     )

#     # Create a new column to categorize grades
#     merged_deliveries_grades['grade_category'] = merged_deliveries_grades['media_final'].apply(
#         lambda x: 'Média Final >= 6' if x >= 6 else 'Média Final < 6'
#     )


#     for class_name, data in merged_deliveries_grades.groupby("class_name"):
#         average_deliveries = data["qtd_entregas"].mean()

#         aprovados = data[data['grade_category'] == 'Média Final >= 6'].shape[0]
#         aprovados_empenhados = data[(data['grade_category'] == 'Média Final >= 6') & (data["qtd_entregas"] > average_deliveries)].shape[0]
#         total_alunos = data["user_id"].nunique()

#         # Plot the relationship between total deliveries and final grades using Plotly Express

#         fig = px.scatter(
#             data,
#             x='qtd_entregas',
#             y='media_final',
#             color="grade_category",
#             title=f'Relationship Between Total Deliveries and Média Final - Turma {class_name}',
#             labels={'qtd_entregas': 'Total Number of Deliveries', 'media_final': 'Média Final'},
#             # trendline="ols",
#             # trendline_scope="overall"
#             color_discrete_map={'Média Final < 6': 'red', 'Média Final >= 6': 'blue'},
#         )
#         fig.add_hline(y=6, line_dash="dash", line_color="black", annotation_text="Média Final >= 6")


#         # total de alunos
#         fig.add_annotation(text=f"Total de alunos: <b>{total_alunos}</b>",
#                             align="left",
#                             showarrow=False,
#                             xref="paper",
#                             yref="paper",
#                             x=1.02,  # Adjust x and y to position it near your legend
#                             y=0.8,
#                             xanchor="left",
#                             font=dict(size=12, color="black"),
#                             bgcolor="rgba(255,255,255,1)"
#                             )
#         # total de aprovados
#         fig.add_annotation(text=f"Total aprovados: {aprovados}",
#                                 align="left",
#                                 showarrow=False,
#                                 xref="paper",
#                                 yref="paper",
#                                 x=1.02,  # Adjust x and y to position it near your legend
#                                 y=0.7,
#                                 xanchor="left",
#                                 font=dict(size=12, color="black"),
#                                 bgcolor="rgba(255,255,255,1)"
#                                 )
#         # total de aprovados que entregaram mais que a média de atividades
#         fig.add_annotation(text=f"Aprovados<br>Empenhados: {aprovados_empenhados}",
#                                 align="left",
#                                 showarrow=False,
#                                 xref="paper",
#                                 yref="paper",
#                                 x=1.02,  # Adjust x and y to position it near your legend
#                                 y=0.55,
#                                 xanchor="left",
#                                 font=dict(size=12, color="black"),
#                                 bgcolor="rgba(255,255,255,1)"
#                                 )
#         # porcentagem de aprovados que entregaram mais que a média de atividades
#         if aprovados > 0:
#             fig.add_annotation(text=f"% empenhados<br> (<b>em aprovados</b>): %{(aprovados_empenhados/aprovados)*100:.2f}",
#                                     align="left",
#                                     showarrow=False,
#                                     xref="paper",
#                                     yref="paper",
#                                     x=1.02,  # Adjust x and y to position it near your legend
#                                     y=0.4,
#                                     xanchor="left",
#                                     font=dict(size=12, color="black"),
#                                     bgcolor="rgba(255,255,255,1)"
#                                     )

#             fig.add_annotation(text=f"% empenhados<br> (<b>no total</b>): %{(aprovados_empenhados/total_alunos)*100:.2f}",
#                                     align="left",
#                                     showarrow=False,
#                                     xref="paper",
#                                     yref="paper",
#                                     x=1.02,  # Adjust x and y to position it near your legend
#                                     y=0.20,
#                                     xanchor="left",
#                                     font=dict(size=12, color="black"),
#                                     bgcolor="rgba(255,255,255,1)"
#                                     )

#         fig.add_vline(x=average_deliveries, line_dash="dash", line_color="black", annotation_text=f"Average Deliveries: {average_deliveries:.2f}")
#         fig.show()
#         fig.write_image(f"relacaoNotasEntregas/Relacao_EntregasNota_Turma{class_name}.png")

# plot_entregasNotasTurma_scatter(df_entregas, "media_final")

# Quantitivo de entregas por tarefa (geral)

In [ ]:
def plot_entregasTarefa_bar(df, requisito, status_tarefa):

    if requisito != "geral":
        df = df[df[requisito] == status_tarefa]

    nome_requisito = "geral" if requisito == "geral" else f"{requisito} = {status_tarefa}"

    qtd_entregas_tarefa = df.groupby(["class_name", "question_name"]).size().to_frame(name="qtd_entregas_tarefa")
    qtd_entregas_tarefa.to_csv("qtd_entregas_questao.csv")
    # Reset the index to make 'class_name' a column
    qtd_entregas_tarefa_geral = qtd_entregas_tarefa.groupby("question_name").sum()
    qtd_entregas_tarefa_reset = qtd_entregas_tarefa_geral.reset_index()
    # Create a scatter plot using Plotly Express
    fig = px.bar(
        qtd_entregas_tarefa_reset,
        x='question_name',
        y='qtd_entregas_tarefa',
        title=f'Quantidade de entregas por questão - {nome_requisito} {detalhamento}',
        labels={'task_question_id': 'ID da Questão da Tarefa', 'qtd_entregas_tarefa': 'Quantidade de Entregas'}
    )

    # Display the plot
    fig.show()
    fig.write_image(f"qtd_entregas/qtdEnqtd_entregas_questao_{requisito}{detalhamento}.png")

def plot_entregasTarefaTurma_bar(df, requisito, status_tarefa):

    df_editado = df.copy()
    if requisito != "geral":
        df_editado = df[df[requisito] == status_tarefa]

    nome_requisito = "geral" if requisito == "geral" else f"{requisito} = {status_tarefa}"
    qtd_entregas_tarefa = df_editado.groupby(["class_name", "question_name"]).size().to_frame(name="qtd_entregas_tarefa")
    # Reset the index to make 'class_name' a column
    qtd_entregas_tarefa_geral = qtd_entregas_tarefa.groupby("question_name").sum()
    qtd_entregas_tarefa_reset = qtd_entregas_tarefa_geral.reset_index()
    for turma, data in qtd_entregas_tarefa.groupby("class_name"):
        data = data.reset_index()
        fig = px.bar(
            data,
            x='question_name',
            y='qtd_entregas_tarefa',
            title=f'Quantidade de entregas por tarefa - {nome_requisito} <br>Turma: {turma} {detalhamento}',
            labels={'task_question_id': 'ID da Questão da Tarefa', 'qtd_entregas_tarefa': 'Quantidade de Entregas'}
        )


        # Display the plot
        fig.show()
        fig.write_image(f"qtd_entregas/qtdEnqtd_entregas_questao_{requisito}{turma}{detalhamento}.png")

# requisito: geral ; grade
plot_entregasTarefa_bar(df_geral, "geral", 1)
plot_entregasTarefaTurma_bar(df_geral, "geral", 1)

NameError: name 'df_geral' is not defined

In [ ]:
def plot_entregasTarefa_stackedBar(df, requisito, max_absences=None):

    if max_absences is not None:
        df = df[df['faltas'] <= max_absences].copy()

    nome_requisito = "geral" if requisito == "geral" else f"{requisito}"

    qtd_entregas_tarefa = df.groupby(["question_name", requisito]).size().to_frame(name="qtd_entregas_tarefa")
    qtd_entregas_tarefa = qtd_entregas_tarefa.reset_index()
    qtd_entregas_tarefa.sort_values("qtd_entregas_tarefa", inplace=True)
    # Create a scatter plot using Plotly Express
    fig = px.bar(
        qtd_entregas_tarefa,
        x='question_name',
        y='qtd_entregas_tarefa',
        color=requisito,
        title=f'Quantidade de entregas por tarefa - {nome_requisito} - Max Absences{max_absences}',
        labels={'question_name': 'Questão da Tarefa', 'qtd_entregas_tarefa': 'Quantidade de Entregas'},
        category_orders={requisito: ["0", '1', '2']} # Set the order of stacked bars
    )

    # Display the plot
    fig.show()
    fig.write_image(f"entregasPorQuestao/{requisito}/qtd_entregas_questao_{requisito}_geral_maxAbsences{max_absences}.png")

def plot_entregasTarefaTurma_stackedBar(df, requisito, max_absences=None):

    df_editado = df.copy()

    if max_absences is not None:
        df_editado = df_editado[df_editado['faltas'] <= max_absences].copy()

    nome_requisito = "geral" if requisito == "geral" else f"{requisito}"
    qtd_entregas_tarefa = df_editado.groupby(["class_name", "question_name", requisito]).size().to_frame(name="qtd_entregas_tarefa")
    # Reset the index to make 'class_name' a column
    qtd_entregas_tarefa_reset = qtd_entregas_tarefa.reset_index()
    for turma, data in qtd_entregas_tarefa_reset.groupby("class_name"):
        fig = px.bar(
            data,
            x='question_name',
            y='qtd_entregas_tarefa',
            color=requisito,
            title=f'Quantidade de entregas por tarefa - {nome_requisito} <br>Turma: {turma} Max Absences{max_absences}',
            labels={'question_name': 'Questão da Tarefa', 'qtd_entregas_tarefa': 'Quantidade de Entregas'}
        )


        # Display the plot
        fig.show()
        fig.write_image(f"entregasPorQuestao/{requisito}/qtd_entregas_questao_{requisito}_turma{turma}_maxAbsences{max_absences}.png")


# requisito: grade ; grade_accounting
plot_entregasTarefa_stackedBar(df_geral.copy(), "grade", max_absences=100)
plot_entregasTarefaTurma_stackedBar(df_geral.copy(), "grade", max_absences=100)


In [ ]:
print("Number of rows in df_geral:", len(df_geral))
print("Number of deliveries with missing question_name:", df_geral['question_name'].isnull().sum())

Number of rows in df_geral: 21761
Number of deliveries with missing question_name: 0


# Qtd entregas x qtd de presenças x notas

# Scatter Plot em grupos

In [ ]:
def plot_entregasNotas_scatter_with_status(df_entregas, df_geral, escopo, max_absences=None):
    # Create copies to avoid modifying original dataframes
    df_entregas_copy = df_entregas.copy()
    df_geral_copy = df_geral.copy()

    # Apply mesclaNotasExameFinal if escopo is 'exame_especial'
    if escopo == "exame_especial":
        df_geral_copy = mesclaNotasExameFinal(df_geral_copy)


    total_deliveries_per_user = df_entregas_copy.groupby("user_id")["qtd_entregas"].sum().reset_index()

    # Merge deliveries, attendance, and final grades
    # Use 'notas_alunos' which already has unique users and their final grades
    merged_df = total_deliveries_per_user.merge(df_geral_copy[['user_id', 'class_name', 'faltas', 'media_final', 'obs']].drop_duplicates(subset=['user_id']), on="user_id", how="left")

    # Filter out students with more than max_absences if the parameter is provided
    if max_absences is not None:
        merged_df = merged_df[merged_df['faltas'] <= max_absences].copy()

    # Categorize students based on status
    def categorize_student(row):
        if pd.isna(row['media_final']) or pd.isna(row['faltas']):
            return 'Dropped'
        elif row['obs'] == 'APROV' and row['faltas'] > 25:
            return 'Approved with Absences' # New category for students approved despite high absences
        elif row['obs'] == 'REPRV' and row['faltas'] > 25:
            return 'Failed by Attendance' # Failed specifically due to high absences
        elif row['media_final'] >= 6 and row['obs'] == 'APROV':
            return 'Approved by Grade' # Approved based on grade
        else:
            return 'Failed by Grade' # Failed based on grade


    merged_df['student_status'] = merged_df.apply(categorize_student, axis=1)

    # Calculate the average of total deliveries for all students who did not drop (for the overall plot)
    average_deliveries_not_dropped_overall = merged_df[merged_df['student_status'] != 'Dropped']['qtd_entregas'].mean()


    # Define marker and color mapping
    status_mapping = {
        'Approved by Grade': {'color': 'blue', 'symbol': 'circle', 'label': 'Aprovados por Nota'},
        'Approved with Absences': {'color': 'dark green', 'symbol': 'square', 'label': 'Aprovados (com Falta)'},
        'Failed by Grade': {'color': 'darkorange', 'symbol': 'circle', 'label': 'Reprovados por Nota'},
        'Failed by Attendance': {'color': 'red', 'symbol': 'x', 'label': 'Reprovados por Falta'},
    }

    # Filter out dropped students for plotting
    merged_df_plotted = merged_df[merged_df['student_status'] != 'Dropped'].copy()


    # Plot for all classes
    # Recalculate the 'student_status_detailed' based on the overall average for the overall plot
    def categorize_approved_students_overall(row):
        if row['student_status'] == "Approved with Absences":
            return "Approved with Absences"
        if row['student_status'] == 'Approved by Grade':
            if row['qtd_entregas'] > average_deliveries_not_dropped_overall:
                return 'Approved by Grade & High Deliveries'
            else:
                return 'Approved by Grade & Low Deliveries'
        else:
            return row['student_status']

    merged_df_plotted['student_status_detailed'] = merged_df_plotted.apply(categorize_approved_students_overall, axis=1)


    fig_all = go.Figure()

    # Update status_mapping to include 'Approved by Grade & High Deliveries' and 'Approved by Grade & Low Deliveries' for the overall plot
    status_mapping_detailed_overall = {
        'Approved by Grade & High Deliveries': {'color': 'green', 'symbol': 'triangle-up', 'label': 'Aprovados por Nota e Empenhados'},
        'Approved by Grade & Low Deliveries': {'color': 'blue', 'symbol': 'circle', 'label': 'Aprovados por Nota (Baixa Entrega)'},
        'Approved with Absences': {'color': 'dark green', 'symbol': 'square', 'label': 'Aprovados (com Falta)'},
        'Failed by Grade': {'color': 'darkorange', 'symbol': 'circle', 'label': 'Reprovados por Nota'},
        'Failed by Attendance': {'color': 'red', 'symbol': 'x', 'label': 'Reprovados por Falta'},
    }


    for status, mapping in status_mapping_detailed_overall.items():
        status_df = merged_df_plotted[merged_df_plotted['student_status_detailed'] == status]
        fig_all.add_trace(go.Scatter(
            x=status_df['qtd_entregas'],
            y=status_df['media_final'],
            mode='markers',
            name=mapping['label'],
            marker=dict(
                color=mapping['color'],
                symbol=mapping['symbol'],
                size=8
            )
        ))

    fig_all.update_layout(
        title=f'Relação entre Total de Entregas e Nota Final (Todas as Turmas) - Max. faltas: {max_absences}',
        xaxis_title='Total de Entregas',
        yaxis_title='Média Final',
        hovermode='closest'
    )

    # Add horizontal line for passing grade
    fig_all.add_hline(y=6, line_dash="dash", line_color="black", annotation_text="Média Final >= 6")

    # Add vertical line for average deliveries
    fig_all.add_vline(x=average_deliveries_not_dropped_overall, line_dash="dash", line_color="black", annotation_text=f"Média de Entregas (Não Reprovados por Falta): {average_deliveries_not_dropped_overall:.2f}")


    # Calculate counts for annotations
    total_students_not_dropped = merged_df_plotted['user_id'].nunique()
    total_approved_grade = merged_df_plotted[merged_df_plotted['student_status'].isin(['Approved by Grade'])]['user_id'].nunique()
    total_approved_absences = merged_df_plotted[merged_df_plotted['student_status'] == 'Approved with Absences']['user_id'].nunique()
    total_approved = total_approved_grade + total_approved_absences
    total_failed_grade = merged_df_plotted[merged_df_plotted['student_status'] == 'Failed by Grade']['user_id'].nunique()
    total_failed_attendance = merged_df_plotted[merged_df_plotted['student_status'] == 'Failed by Attendance']['user_id'].nunique()
    total_failed = total_failed_grade + total_failed_attendance
    approved_high_deliveries = merged_df_plotted[merged_df_plotted['student_status_detailed'] == 'Approved by Grade & High Deliveries']['user_id'].nunique()


    # Add annotations
    fig_all.add_annotation(text=f"Total de alunos (sem abandono): <b>{total_students_not_dropped}</b>",
                            align="left", showarrow=False, xref="paper", yref="paper", x=1.02, y=0.60, xanchor="left", font=dict(size=12, color="black"), bgcolor="rgba(255,255,255,1)")
    fig_all.add_annotation(text=f"Total aprovados: {total_approved}",
                            align="left", showarrow=False, xref="paper", yref="paper", x=1.02, y=0.55, xanchor="left", font=dict(size=12, color="black"), bgcolor="rgba(255,255,255,1)")
    fig_all.add_annotation(text=f"Total reprovados por nota: {total_failed_grade}",
                            align="left", showarrow=False, xref="paper", yref="paper", x=1.02, y=0.45, xanchor="left", font=dict(size=12, color="black"), bgcolor="rgba(255,255,255,1)")
    fig_all.add_annotation(text=f"Total reprovados por falta: {total_failed_attendance}",
                            align="left", showarrow=False, xref="paper", yref="paper", x=1.02, y=0.35, xanchor="left", font=dict(size=12, color="black"), bgcolor="rgba(255,255,255,1)")
    fig_all.add_annotation(text=f"Total reprovados: {total_failed}",
                            align="left", showarrow=False, xref="paper", yref="paper", x=1.02, y=0.25, xanchor="left", font=dict(size=12, color="black"), bgcolor="rgba(255,255,255,1)")
    fig_all.add_annotation(text=f"Aprovados e Empenhados: {approved_high_deliveries}",
                            align="left", showarrow=False, xref="paper", yref="paper", x=1.02, y=0.15, xanchor="left", font=dict(size=12, color="black"), bgcolor="rgba(255,255,255,1)")
    if total_approved > 0:
        fig_all.add_annotation(text=f"% Empenhados (em aprovados por nota): %{(approved_high_deliveries/total_approved_grade)*100:.2f}",
                                align="left", showarrow=False, xref="paper", yref="paper", x=1.02, y=0.10, xanchor="left", font=dict(size=12, color="black"), bgcolor="rgba(255,255,255,1)")
        fig_all.add_annotation(text=f"% Empenhados (no total): %{(approved_high_deliveries/total_students_not_dropped)*100:.2f}",
                                align="left", showarrow=False, xref="paper", yref="paper", x=1.02, y=0.05, xanchor="left", font=dict(size=12, color="black"), bgcolor="rgba(255,255,255,1)")
    # Add annotation for Approved with Absences in the overall plot
    fig_all.add_annotation(text=f"Aprovados (com Falta): {total_approved_absences}",
                            align="left", showarrow=False, xref="paper", yref="paper", x=1.02, y=0.50, xanchor="left", font=dict(size=12, color="black"), bgcolor="rgba(255,255,255,1)")


    fig_all.show()
    fig_all.write_image(f"relacaoNotasEntregas/{max_absences}_DivisaoPorGrupos_EntregasNota_Status_all_classes{detalhamento}.png", width=2280, height=1140 )


    # Plot for each class
    for class_name, data in merged_df_plotted.groupby("class_name"):
        average_deliveries_class = data['qtd_entregas'].mean()

        # Recalculate 'student_status_detailed' based on the class average for class-specific plots
        def categorize_approved_students_class(row):
            if row['student_status'] == "Approved with Absences":
                return "Approved with Absences"
            if row['student_status'] == 'Approved by Grade':
                if row['qtd_entregas'] > average_deliveries_class:
                    return 'Approved by Grade & High Deliveries'
                else:
                    return 'Approved by Grade & Low Deliveries'
            else:
                return row['student_status']

        data['student_status_detailed'] = data.apply(categorize_approved_students_class, axis=1)

        fig_class = go.Figure()

        # Update status_mapping to include 'Approved by Grade & High Deliveries' and 'Approved by Grade & Low Deliveries' for the class plots
        status_mapping_detailed_class = {
            'Approved by Grade & High Deliveries': {'color': 'green', 'symbol': 'triangle-up', 'label': 'Aprovados por Nota e Empenhados'},
            'Approved by Grade & Low Deliveries': {'color': 'blue', 'symbol': 'circle', 'label': 'Aprovados por Nota (Baixa Entrega)'},
            'Approved with Absences': {'color': 'dark green', 'symbol': 'square', 'label': 'Aprovados (com Falta)'},
            'Failed by Grade': {'color': 'darkorange', 'symbol': 'circle', 'label': 'Reprovados por Nota'},
            'Failed by Attendance': {'color': 'red', 'symbol': 'x', 'label': 'Reprovados por Falta'},
        }

        for status, mapping in status_mapping_detailed_class.items():
            status_df_class = data[data['student_status_detailed'] == status]
            fig_class.add_trace(go.Scatter(
                x=status_df_class['qtd_entregas'],
                y=status_df_class['media_final'],
                mode='markers',
                name=mapping['label'],
                marker=dict(
                    color=mapping['color'],
                    symbol=mapping['symbol'],
                    size=8
                )
            ))

        fig_class.update_layout(
            title=f'Relação entre Total de Entregas e Nota Final - Turma {class_name} - Max. faltas: {max_absences}',
            xaxis_title='Total de Entregas',
            yaxis_title='Média Final',
            hovermode='closest'
        )

        fig_class.add_hline(y=6, line_dash="dash", line_color="black", annotation_text="Média Final >= 6")
        fig_class.add_vline(x=average_deliveries_class, line_dash="dash", line_color="black", annotation_text=f"Média de Entregas: {average_deliveries_class:.2f}")

        # Calculate counts for annotations for the current class
        total_students_class = data['user_id'].nunique()
        total_approved_grade_class = data[data['student_status'].isin(['Approved by Grade'])]['user_id'].nunique()
        total_approved_absences_class = data[data['student_status'] == 'Approved with Absences']['user_id'].nunique()
        total_approved_class = total_approved_grade_class + total_approved_absences_class
        total_failed_grade_class = data[data['student_status'] == 'Failed by Grade']['user_id'].nunique()
        total_failed_attendance_class = data[data['student_status'] == 'Failed by Attendance']['user_id'].nunique()
        total_failed_class = total_failed_grade_class + total_failed_attendance_class
        approved_high_deliveries_class = data[data['student_status_detailed'] == 'Approved by Grade & High Deliveries']['user_id'].nunique()


        # Add annotations for the current class
        fig_class.add_annotation(text=f"Total de alunos (sem abandono): <b>{total_students_class}</b>",
                                align="left", showarrow=False, xref="paper", yref="paper", x=1.02, y=0.60, xanchor="left", font=dict(size=12, color="black"), bgcolor="rgba(255,255,255,1)")
        fig_class.add_annotation(text=f"Total aprovados: {total_approved_class}",
                                align="left", showarrow=False, xref="paper", yref="paper", x=1.02, y=0.55, xanchor="left", font=dict(size=12, color="black"), bgcolor="rgba(255,255,255,1)")
        fig_class.add_annotation(text=f"Total reprovados por nota: {total_failed_grade_class}",
                                align="left", showarrow=False, xref="paper", yref="paper", x=1.02, y=0.45, xanchor="left", font=dict(size=12, color="black"), bgcolor="rgba(255,255,255,1)")
        fig_class.add_annotation(text=f"Total reprovados por falta: {total_failed_attendance_class}",
                                align="left", showarrow=False, xref="paper", yref="paper", x=1.02, y=0.35, xanchor="left", font=dict(size=12, color="black"), bgcolor="rgba(255,255,255,1)")
        fig_class.add_annotation(text=f"Total reprovados: {total_failed_class}",
                                align="left", showarrow=False, xref="paper", yref="paper", x=1.02, y=0.25, xanchor="left", font=dict(size=12, color="black"), bgcolor="rgba(255,255,255,1)")
        fig_class.add_annotation(text=f"Aprovados e Empenhados: {approved_high_deliveries_class}",
                                align="left", showarrow=False, xref="paper", yref="paper", x=1.02, y=0.15, xanchor="left", font=dict(size=12, color="black"), bgcolor="rgba(255,255,255,1)")
        if total_approved_class > 0:
            fig_class.add_annotation(text=f"% Empenhados (em aprovados por nota): %{(approved_high_deliveries_class/total_approved_grade_class)*100:.2f}",
                                    align="left", showarrow=False, xref="paper", yref="paper", x=1.02, y=0.10, xanchor="left", font=dict(size=12, color="black"), bgcolor="rgba(255,255,255,1)")
            fig_class.add_annotation(text=f"% Empenhados (no total): %{(approved_high_deliveries_class/total_students_class)*100:.2f}",
                                    align="left", showarrow=False, xref="paper", yref="paper", x=1.02, y=0.05, xanchor="left", font=dict(size=12, color="black"), bgcolor="rgba(255,255,255,1)")

        # Add annotation for Approved with Absences in the class plot
        fig_class.add_annotation(text=f"Aprovados (com Falta): {total_approved_absences_class}",
                                align="left", showarrow=False, xref="paper", yref="paper", x=1.02, y=0.50, xanchor="left", font=dict(size=12, color="black"), bgcolor="rgba(255,255,255,1)")


        fig_class.show()
        fig_class.write_image(f"relacaoNotasEntregas/{max_absences}_DivisaoPorGrupos_EntregasNota_Status_Turma{class_name}{detalhamento}.png", width=1140, height=570)


# Call the new function
# plot_entregasNotas_scatter_with_status(df_entregas, df_geral, "exame_especial") # Example call without max_absences filter
# Example call with a max_absences filter (e.g., excluding students with more than 10 absences)
plot_entregasNotas_scatter_with_status(df_entregas, df_geral, "exame_especial", max_absences=35)

In [ ]:
print(f"Students with NaN absences in original data: {df_geral[df_geral['faltas'].isna()]['user_id'].nunique()}")
print(f"Total users with faltas field empty: {df_geral['faltas'].isnull().sum()}")

Students with NaN absences in original data: 0
Total users with faltas field empty: 0


# Gráficos como os do artigo

In [ ]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go

def _prepare_data(df):
    """
    Performs common data preparation: timestamp conversion and filters out late deliveries.
    This function is now a base for both grade and SSR calculations.
    """
    df_copy = df.copy()

    # Convert necessary columns to datetime format
    df_copy['delivery_timestamp'] = pd.to_datetime(df_copy['delivery_timestamp'])
    df_copy['task_starting_timestamp'] = pd.to_datetime(df_copy['task_starting_timestamp'])
    df_copy['task_finishing_timestamp'] = pd.to_datetime(df_copy['task_finishing_timestamp'])

    # Filter out late deliveries
    df_copy = df_copy[df_copy['delivery_timestamp'] <= df_copy['task_finishing_timestamp']].copy()

    # Sort by user, task, question, and delivery time for first/last/max logic
    df_sorted = df_copy.sort_values(by=['user_id', 'task_name', 'question_name', 'delivery_timestamp'])

    return df_sorted

def _calculate_grades_and_ssr(df_sorted):
    """
    Calculates average grades (first, last, max) and average SSR (first, last)
    per question, based on the sorted dataframe.
    """
    group_keys = ['task_name', 'question_name']
    user_question_keys = ['user_id', 'task_name', 'question_name']

    # Calculate average deliveries per student
    deliveries_per_student = df_sorted.groupby(user_question_keys).size().to_frame(name='deliveries_count').reset_index()
    avg_deliveries_per_question = deliveries_per_student.groupby(group_keys)['deliveries_count'].mean().to_frame(name='avg_deliveries_per_student').reset_index()


    # Calculate average grade of first submission
    first_grades = df_sorted.groupby(user_question_keys)['grade'].first().reset_index()
    avg_first_submission_grade_per_question = first_grades.groupby(group_keys)['grade'].mean().reset_index()
    avg_first_submission_grade_per_question.rename(columns={'grade': 'avg_first_submission_grade'}, inplace=True)

    # Calculate average grade of last submission
    last_grades = df_sorted.groupby(user_question_keys)['grade'].last().reset_index()
    avg_last_submission_grade_per_question = last_grades.groupby(group_keys)['grade'].mean().reset_index()
    avg_last_submission_grade_per_question.rename(columns={'grade': 'avg_last_submission_grade'}, inplace=True)

    # Calculate average max grade
    max_grades = df_sorted.groupby(user_question_keys)['grade'].max().reset_index()
    avg_max_submission_grade_per_question = max_grades.groupby(group_keys)['grade'].mean().reset_index()
    avg_max_submission_grade_per_question.rename(columns={'grade': 'avg_max_submission_grade'}, inplace=True)

    # Calculate SSR (Submission Speed Ratio) for all deliveries within the deadline
    df_ssr = df_sorted.copy() # Use all deliveries within the deadline for SSR calculation


    if df_ssr.empty:
        # Return empty dataframes if no qualifying deliveries for SSR
        empty_ssr_first = pd.DataFrame(columns=group_keys + ['avg_first_submission_ssr'])
        empty_ssr_last = pd.DataFrame(columns=group_keys + ['avg_last_submission_ssr'])
        return (avg_deliveries_per_question, avg_first_submission_grade_per_question,
                avg_last_submission_grade_per_question, avg_max_submission_grade_per_question,
                empty_ssr_first, empty_ssr_last)


    df_ssr['numerator'] = (df_ssr['delivery_timestamp'] - df_ssr['task_starting_timestamp']).dt.total_seconds()
    df_ssr['denominator'] = (df_ssr['task_finishing_timestamp'] - df_ssr['task_starting_timestamp']).dt.total_seconds()

    df_ssr['SSR'] = np.where(
        df_ssr['denominator'] == 0,
        np.nan,
        df_ssr['numerator'] / df_ssr['denominator']
    )

    df_ssr.dropna(subset=['SSR'], inplace=True)

    # Calculate average SSR of first submission
    first_ssr = df_ssr.groupby(user_question_keys)['SSR'].first().reset_index()
    avg_first_submission_ssr_per_question = first_ssr.groupby(group_keys)['SSR'].mean().reset_index()
    avg_first_submission_ssr_per_question.rename(columns={'SSR': 'avg_first_submission_ssr'}, inplace=True)

    # Calculate average SSR of last submission
    last_ssr = df_ssr.groupby(user_question_keys)['SSR'].last().reset_index()
    avg_last_submission_ssr_per_question = last_ssr.groupby(group_keys)['SSR'].mean().reset_index()
    avg_last_submission_ssr_per_question.rename(columns={'SSR': 'avg_last_submission_ssr'}, inplace=True)


    return (avg_deliveries_per_question, avg_first_submission_grade_per_question,
            avg_last_submission_grade_per_question, avg_max_submission_grade_per_question,
            avg_first_submission_ssr_per_question, avg_last_submission_ssr_per_question)


def _create_mapping_table(merged_analysis, file_prefix):
    """Generates the Plotly table that maps the X-axis number to the question name."""

    # Extract the mapping columns
    mapping_df = merged_analysis[['x_number', 'question_name']].drop_duplicates().sort_values(by='x_number')

    # Create the Plotly table
    fig_table = go.Figure(data=[go.Table(
        header=dict(values=['Número da Questão (X-Axis)', 'Nome da Questão'],
                    fill_color='paleturquoise',
                    align='left'),
        cells=dict(values=[mapping_df['x_number'].tolist(), mapping_df['question_name'].tolist()],
                   fill_color='lavender',
                   align='left'))
    ])

    fig_table.update_layout(title=f"Mapeamento de Questões - {file_prefix}")
    fig_table.show()
    fig_table.write_image(f"entregas_notas_SSR/Tabela_Questoes.png")


def _create_figure(merged_analysis, title_suffix, file_prefix, use_max_grade=False):
    """
    Creates and configures the Plotly figure including deliveries, grades, and SSR.
    """

    # Robust categorical ordering for question_name (needed for the table mapping)
    question_order_df = merged_analysis[['task_name', 'question_name']].drop_duplicates().sort_values(by='task_name')
    question_order = question_order_df['question_name'].tolist()

    merged_analysis['question_name'] = pd.Categorical(
        merged_analysis['question_name'],
        categories=question_order,
        ordered=True
    )
    merged_analysis = merged_analysis.sort_values('question_name').reset_index(drop=True)

    # Create the numbered X-axis column
    merged_analysis['x_number'] = merged_analysis.index + 1

    # Apply x10 scaling to grades (0-10 -> 0-100)
    # merged_analysis['avg_first_submission_grade_scaled'] = merged_analysis['avg_first_submission_grade'] * 10
    # merged_analysis['avg_last_submission_grade_scaled'] = merged_analysis['avg_last_submission_grade'] * 10
    # merged_analysis['avg_max_submission_grade_scaled'] = merged_analysis['avg_max_submission_grade'] * 10

    # Apply x100 scaling to SSR (0-1 -> 0-100)
    merged_analysis['avg_first_submission_ssr_scaled'] = merged_analysis['avg_first_submission_ssr'] * 100
    merged_analysis['avg_last_submission_ssr_scaled'] = merged_analysis['avg_last_submission_ssr'] * 100

    fig = go.Figure()

    # 1. Average Deliveries Labels (Y1) - One decimal place
    delivery_labels = merged_analysis['avg_deliveries_per_student'].round(1).astype(str)

    # Add bars for average deliveries per student (Y1-axis)
    fig.add_trace(go.Bar(
        x=merged_analysis['x_number'],
        y=merged_analysis['avg_deliveries_per_student'],
        name='Média de Entregas por Aluno',
        yaxis='y1',
        marker_color='gray',
        text=delivery_labels,
        textposition='none'
    ))

    # 2. Grade Labels (Y2) - One decimal place
    grade_first_labels = merged_analysis['avg_first_submission_grade'].round(1).astype(str)

    # Add line for average first submission grade (Y2-axis)
    fig.add_trace(go.Scatter(
        x=merged_analysis['x_number'],
        y=merged_analysis['avg_first_submission_grade'],
        mode='lines+markers+text', # Added +text mode
        name='Nota média da primeira submissão (0-10)',
        yaxis='y2',
        line=dict(color='red'),
        text=grade_first_labels, # Add text labels
        textposition='bottom center', # Position for visibility
        textfont=dict(
            color='black', # Optional: Choose a color that stands out
            weight='bold'  # Sets the font to bold
        )
    ))

    # Add line for average last/max submission grade (Y2-axis)
    grade_col_to_plot = 'avg_max_submission_grade' if use_max_grade else 'avg_last_submission_grade'
    grade_name = 'Nota média da maior submissão (0-10)' if use_max_grade else 'Nota média da última submissão (0-10)'
    grade_last_labels = merged_analysis[grade_col_to_plot].round(1).astype(str)

    fig.add_trace(go.Scatter(
        x=merged_analysis['x_number'],
        y=merged_analysis[grade_col_to_plot],
        mode='lines+markers+text', # Added +text mode
        name=grade_name,
        yaxis='y2',
        line=dict(color='green'),
        text=grade_last_labels, # Add text labels
        textposition='top center', # Position for visibility
        textfont=dict(
            color='black', # Optional: Choose a color that stands out
            weight='bold'  # Sets the font to bold
        )
    ))

    # 3. SSR Labels (Y2) - One decimal place
    ssr_first_labels = merged_analysis['avg_first_submission_ssr_scaled'].round(1).astype(str)

    # Add line for average first submission SSR (Y2-axis)
    fig.add_trace(go.Scatter(
        x=merged_analysis['x_number'],
        y=merged_analysis['avg_first_submission_ssr_scaled'],
        mode='lines+markers+text', # Added +text mode
        name='Média do SSR da primeira submissão (0-100%)',
        yaxis='y2',
        line=dict(color='blue'),
        text=ssr_first_labels, # Add text labels
        textposition='bottom center', # Position for visibility
        textfont=dict(
            color='black', # Optional: Choose a color that stands out
            weight='bold'  # Sets the font to bold
        )
    ))

    # Add line for average last submission SSR (Y2-axis)
    ssr_last_labels = merged_analysis['avg_last_submission_ssr_scaled'].round(1).astype(str)

    fig.add_trace(go.Scatter(
        x=merged_analysis['x_number'],
        y=merged_analysis['avg_last_submission_ssr_scaled'],
        mode='lines+markers+text', # Added +text mode
        name='Média do SSR da última submissão (0-100%)',
        yaxis='y2',
        line=dict(color='purple'),
        text=ssr_last_labels, # Add text labels
        textposition='top center', # Position for visibility
        textfont=dict(
            color='black', # Optional: Choose a color that stands out
            weight='bold'  # Sets the font to bold
        )
    ))


    # Update layout for dual y-axes and visual improvements
    fig.update_layout(
        title=f'Análise por Questão: Média de Entregas, Notas Médias e SSR Médio - {title_suffix}',
        width=1900,
        height=950, # Increased Plot Height
        margin=dict(l=50, r=50, b=150, t=50),

        xaxis=dict(
            title='Número da Questão',
            tickangle=0,
            tickfont=dict(size=12),
            dtick=5,
            tick0=1,
        ),
        # Y1 axis for Average Deliveries
        yaxis=dict(
            title=dict(text='Média de Entregas por Aluno', font=dict(color='gray')),
            tickfont=dict(color='gray'),
            # range=[0, merged_analysis['avg_deliveries_per_student'].max() * 2.0]
            range=[0, 10]
        ),
        # Y2 axis for Grades (0-100) and SSR (0-100) - Shared scale
        yaxis2=dict(
            title=dict(text='Nota / SSR (%)', font=dict(color='black')),
            tickfont=dict(color='black'),
            overlaying='y',
            side='right',
            range=[0, 105]
        ),
        legend=dict(
            x=0.5,
            y=-0.2,
            xanchor='center',
            yanchor='top',
            orientation='h'
        ),
        barmode='group'
    )

    fig.show()
    fig.write_image(f"entregas_notas_SSR/{file_prefix}.png", width=2280, height=1140)

    # Generate the mapping table after the figure
    _create_mapping_table(merged_analysis, file_prefix)


# ----------------------------------------------------------------------------------
# COMBINED PLOT FOR ALL CLASSES
# ----------------------------------------------------------------------------------

def plot_question_analysis_combined(df, use_max_grade=False):
    """
    Generates a single dual-axis plot for all classes combined, including
    average deliveries, grades, and SSR.
    """
    df_sorted = _prepare_data(df)
    if df_sorted.empty:
        print("No data available after initial filtering.")
        return

    (avg_deliveries, avg_first_grade, avg_last_grade, avg_max_grade,
     avg_first_ssr, avg_last_ssr) = _calculate_grades_and_ssr(df_sorted)

    # Merge all calculated averages
    merged_analysis = avg_deliveries.merge(avg_first_grade, on=['task_name', 'question_name'], how='left')
    merged_analysis = merged_analysis.merge(avg_last_grade, on=['task_name', 'question_name'], how='left')
    merged_analysis = merged_analysis.merge(avg_max_grade, on=['task_name', 'question_name'], how='left')
    merged_analysis = merged_analysis.merge(avg_first_ssr, on=['task_name', 'question_name'], how='left')
    merged_analysis = merged_analysis.merge(avg_last_ssr, on=['task_name', 'question_name'], how='left')


    # Fill NaN SSR values with 0 (or another appropriate value) if a question had no qualifying submissions for SSR
    merged_analysis['avg_first_submission_ssr'] = merged_analysis['avg_first_submission_ssr'].fillna(0)
    merged_analysis['avg_last_submission_ssr'] = merged_analysis['avg_last_submission_ssr'].fillna(0)


    # Create the figure using the helper function
    _create_figure(
        merged_analysis,
        title_suffix="Todas as Turmas (Análise Global)",
        file_prefix="analise_por_questao_global - Completa",
        use_max_grade=use_max_grade
    )

# ----------------------------------------------------------------------------------
# ORIGINAL FUNCTION: PLOT PER CLASS
# ----------------------------------------------------------------------------------

def plot_question_analysis_per_class(df, use_max_grade=False):
    """
    Generates a dual-axis plot for each class, including average deliveries,
    grades, and SSR.
    """

    df_sorted = _prepare_data(df)
    if df_sorted.empty:
        print("No data available after initial filtering.")
        return

    detalhamento = "_analise_completa" # Use this placeholder for file naming


    for class_name, data in df_sorted.groupby("class_name"):

        (avg_deliveries, avg_first_grade, avg_last_grade, avg_max_grade,
         avg_first_ssr, avg_last_ssr) = _calculate_grades_and_ssr(data)

        if avg_deliveries.empty and avg_first_grade.empty and avg_last_grade.empty and avg_max_grade.empty and avg_first_ssr.empty and avg_last_ssr.empty:
             print(f"No data available for class {class_name} after filtering.")
             continue


        # Merge all calculated averages
        merged_analysis = avg_deliveries.merge(avg_first_grade, on=['task_name', 'question_name'], how='left')
        merged_analysis = merged_analysis.merge(avg_last_grade, on=['task_name', 'question_name'], how='left')
        merged_analysis = merged_analysis.merge(avg_max_grade, on=['task_name', 'question_name'], how='left')
        merged_analysis = merged_analysis.merge(avg_first_ssr, on=['task_name', 'question_name'], how='left')
        merged_analysis = merged_analysis.merge(avg_last_ssr, on=['task_name', 'question_name'], how='left')

        # Fill NaN SSR values with 0 (or another appropriate value) if a question had no qualifying submissions for SSR
        merged_analysis['avg_first_submission_ssr'] = merged_analysis['avg_first_submission_ssr'].fillna(0)
        merged_analysis['avg_last_submission_ssr'] = merged_analysis['avg_last_submission_ssr'].fillna(0)

        # Create the figure using the helper function
        _create_figure(
            merged_analysis,
            title_suffix=f"Turma {class_name}{detalhamento}",
            file_prefix=f"analise_por_questao_turma_{class_name.replace(' ', '_')} - Completa",
            use_max_grade=use_max_grade
        )

In [ ]:
# Call the function
plot_question_analysis_combined(df_geral, use_max_grade=True)
plot_question_analysis_per_class(df_geral, use_max_grade=True)

# heatmaps

In [ ]:
def plot_deliveries_attendance_grade_heatmap_all_classes(df_entregas, df_geral):
    # Create copies to avoid modifying original dataframes
    df_entregas_copy = df_entregas.copy()
    df_geral_copy = df_geral.copy()

    # Calculate total deliveries per user across all classes
    total_deliveries_per_user = df_entregas_copy.groupby("user_id")["qtd_entregas"].sum().reset_index()

    # Calculate attendance (72 - faltas) for each user (assuming 'faltas' is unique per user in df_geral)
    # Drop duplicates to ensure one row per user for attendance
    df_attendance = df_geral_copy.drop_duplicates(subset=['user_id'])[['user_id', 'faltas']]
    df_attendance['attendance'] = 100 - df_attendance['faltas']

    # Merge deliveries, attendance, and final grades
    # Use 'notas_alunos' which already has unique users and their final grades
    merged_df = total_deliveries_per_user.merge(df_attendance, on="user_id", how="left")
    merged_df = merged_df.merge(notas_alunos[["user_id", "media_final"]], on="user_id", how="left")

    # Drop rows with missing media_final, qtd_entregas or attendance
    merged_df.dropna(subset=['media_final', 'qtd_entregas', 'attendance'], inplace=True)

    # Create bins for heatmap
    deliveries_bins, delivery_bin_edges = pd.cut(merged_df['qtd_entregas'], bins=10, labels=False, retbins=True)
    attendance_bins, attendance_bin_edges = pd.cut(merged_df['attendance'], bins=10, labels=False, retbins=True)

    # Create labels for the bins
    delivery_labels = [f'{int(delivery_bin_edges[i])}-{int(delivery_bin_edges[i+1])}' for i in range(len(delivery_bin_edges)-1)]
    attendance_labels = [f'{int(attendance_bin_edges[i])}-{int(attendance_bin_edges[i+1])}' for i in range(len(attendance_bin_edges)-1)]


    heatmap_data = merged_df.groupby([deliveries_bins, attendance_bins])['media_final'].mean().unstack()

    fig = go.Figure(data=go.Heatmap(
        z=heatmap_data.values,
        x=delivery_labels, # Use bin labels for x-axis
        y=attendance_labels, # Use bin labels for y-axis
        colorscale='Bluered_r',
        colorbar=dict(title='Nota média final')
    ))

    fig.update_layout(
        title=f'Heatmap da média final: Entregas vs Presença (Todas as turmas){detalhamento}',
        xaxis_title='Intervalos de entrega',
        yaxis_title='Intervalos de presença (%)'
    )

    fig.show()
    fig.write_image(f"relacaoNotasEntregas/heatmap_entregas_attendance_grade_all_classes{detalhamento}.png")


def plot_deliveries_attendance_grade_heatmap_per_class(df_entregas, df_geral):
    # Create copies to avoid modifying original dataframes
    df_entregas_copy = df_entregas.copy()
    df_geral_copy = df_geral.copy()

    # Calculate total deliveries per user per class
    total_deliveries_per_user_and_class = df_entregas_copy.groupby(["class_name", "user_id"])["qtd_entregas"].sum().reset_index()

    # Calculate attendance (72 - faltas) for each user per class
    # Drop duplicates to ensure one row per user per class for attendance
    df_attendance = df_geral_copy.drop_duplicates(subset=['class_name', 'user_id'])[['class_name', 'user_id', 'faltas']]
    df_attendance['attendance'] = 100 - df_attendance['faltas']


    # Merge deliveries, attendance, and final grades per class
    merged_df = total_deliveries_per_user_and_class.merge(df_attendance, on=["class_name", "user_id"], how="left")
    merged_df = merged_df.merge(notas_alunos[["user_id", "media_final"]], on="user_id", how="left")

    # Drop rows with missing media_final, qtd_entregas or attendance
    merged_df.dropna(subset=['media_final', 'qtd_entregas', 'attendance'], inplace=True)


    for class_name, data in merged_df.groupby("class_name"):
        # Create bins for heatmap for each class
        deliveries_bins, delivery_bin_edges = pd.cut(data['qtd_entregas'], bins=10, labels=False, retbins=True) # Fewer bins for potentially smaller class sizes
        attendance_bins, attendance_bin_edges = pd.cut(data['attendance'], bins=10, labels=False, retbins=True)

        # Create labels for the bins
        delivery_labels = [f'{int(delivery_bin_edges[i])}-{int(delivery_bin_edges[i+1])}' for i in range(len(delivery_bin_edges)-1)]
        attendance_labels = [f'{int(attendance_bin_edges[i])}-{int(attendance_bin_edges[i+1])}' for i in range(len(attendance_bin_edges)-1)]


        heatmap_data = data.groupby([deliveries_bins, attendance_bins])['media_final'].mean().unstack()

        fig = go.Figure(data=go.Heatmap(
            z=heatmap_data.values,
            x=delivery_labels, # Use bin labels for x-axis
            y=attendance_labels, # Use bin labels for y-axis
            colorscale='bluered_r',
            colorbar=dict(title='Nota média final')
        ))

        fig.update_layout(
            title=f'Heatmap da média final: Entregas vs Presença - Turma {class_name}{detalhamento}',
            xaxis_title='Intervalos de entrega',
            yaxis_title='Intervalos de presença'
        )

        fig.show()
        fig.write_image(f"relacaoNotasEntregas/heatmap_entregas_attendance_grade_turma{class_name}{detalhamento}.png")

plot_deliveries_attendance_grade_heatmap_all_classes(df_entregas, df_geral)
plot_deliveries_attendance_grade_heatmap_per_class(df_entregas, df_geral)

# Moodle Data Analytics

In [ ]:
df_moodle = pd.read_csv('moodle_data_merged.csv')

FileNotFoundError: [Errno 2] No such file or directory: 'moodle_data_merged.csv'

In [ ]:
df_moodle

# Exportação de Dados

In [ ]:
# Compacta os gráficos de totais de atividades realizadas por aluno (para que o arquivo possa ser baixado)
!zip -r tabelasTotais.zip tabelasTotais

!zip -r qtdEntregas.zip qtd_entregas

!zip -r relacaoNotasEntregas.zip relacaoNotasEntregas

!zip -r entregasPorQuestao.zip entregasPorQuestao

!zip -r statusFinal.zip statusFinal

!zip -r statusEntregas.zip statusEntregas

!zip -r entregas_notas_SSR.zip entregas_notas_SSR

!zip -r boxPlot_notas.zip boxPlot_notas

updating: tabelasTotais/ (stored 0%)
updating: tabelasTotais/TotalEntregasUsuario_Turma_5_6_(11)_(24.2).png (deflated 19%)
updating: tabelasTotais/TotalEntregasUsuario_Turma_3_4_(24.2).png (deflated 16%)
updating: tabelasTotais/TotalEntregasUsuario_Turma_1_2_(24.2).png (deflated 19%)
updating: tabelasTotais/TotalEntregasUsuario_Turma_11_12_(24.2).png (deflated 16%)
updating: tabelasTotais/TotalEntregasUsuario_Turma_13_14_(24.2).png (deflated 16%)
updating: tabelasTotais/TotalEntregasUsuario_Turma_17_18_(24.2).png (deflated 18%)
updating: tabelasTotais/TotalEntregasUsuario_Turma_9_10_(24.2).png (deflated 15%)
updating: tabelasTotais/TotalEntregasUsuario_Turma_15_16_(24.2).png (deflated 16%)
updating: tabelasTotais/TotalEntregasUsuario_Turma_19_20_(24.2).png (deflated 16%)
updating: tabelasTotais/TotalEntregasUsuario_Turma_7_8_(24.2).png (deflated 19%)
updating: qtd_entregas/ (stored 0%)
updating: qtd_entregas/qtdEnqtd_entregas_questao_geral17_18 (24.2)_(comReprFaltas).png (deflated 11%)

Maior nota das entregas VS nota da última entrega

In [ ]:
# import pandas as pd
# from google.colab import files
# import io

# # --- Configurações ---
# NOME_ARQUIVO_ENTRADA = "full_data_merged_classesRevised_comReprFaltas.csv"
# NOME_ARQUIVO_SAIDA = "notas_consolidadas_final.csv"
# DELIMITADOR = ';'

# # Colunas-chave para identificação única de Aluno e Questão
# KEY_COLS_AGREGACAO = ['user_id', 'task_question_id', 'task_name', 'question_name']

# # --- 1. Carregar o arquivo ---
# print("--- 1. Iniciando carregamento do arquivo ---")
# try:
#     # Tenta ler o arquivo diretamente
#     df = pd.read_csv(NOME_ARQUIVO_ENTRADA, sep=DELIMITADOR)
# except FileNotFoundError:
#     print(f"O arquivo '{NOME_ARQUIVO_ENTRADA}' não foi encontrado. Por favor, faça o upload agora.")

#     # Solicita o upload (substitua esta linha pela sua forma de carregamento se estiver fora do Colab)
#     uploaded = files.upload()

#     if not uploaded:
#         print("Upload cancelado ou falhou. Terminando script.")
#         exit()

#     # Lógica para ler o arquivo carregado
#     nome_arquivo_lido = list(uploaded.keys())[0]
#     content = uploaded[nome_arquivo_lido]
#     print(f"Atenção: Usando o arquivo carregado: '{nome_arquivo_lido}'.")
#     try:
#         df = pd.read_csv(io.BytesIO(content), sep=DELIMITADOR)
#     except Exception:
#         try:
#             df = pd.read_csv(io.BytesIO(content), sep=DELIMITADOR, encoding='latin1')
#         except Exception as e:
#             print(f"Falha ao ler o arquivo. Erro: {e}")
#             exit()
# except Exception as e:
#     print(f"Erro inesperado ao carregar o arquivo: {e}")
#     exit()

# print(f"Dados lidos com sucesso. Total de {len(df)} linhas.")


# # --- 2. Pré-processamento e Limpeza ---
# print("--- 2. Pré-processando dados ---")

# try:
#     # Trata as colunas necessárias
#     df['grade'] = pd.to_numeric(df['grade'], errors='coerce')
#     df['delivery_timestamp'] = pd.to_datetime(df['delivery_timestamp'], errors='coerce')
# except KeyError as e:
#     print(f"\nERRO: Uma coluna essencial não foi encontrada: {e}")
#     print("Verifique se as colunas 'user_id', 'task_question_id', 'task_name', 'question_name', 'grade' e 'delivery_timestamp' existem no seu arquivo.")
#     exit()

# # Remove linhas com dados inválidos essenciais
# df_limpo = df.dropna(subset=['grade', 'delivery_timestamp'] + KEY_COLS_AGREGACAO).copy()
# if len(df) != len(df_limpo):
#     print(f"Foram removidas {len(df) - len(df_limpo)} linhas com dados inválidos.")


# # --- 3. Agregação: Maior Nota e Último Timestamp (Corrigindo o KeyError) ---
# print("--- 3. Agregando Maior Nota e Última Entrega (Corrigido) ---")

# # 3a. Calcular a Maior Nota e o Último Timestamp
# df_agregado = df_limpo.groupby(KEY_COLS_AGREGACAO).agg(
#     # A maior nota de todas as entregas
#     Maior_Nota_das_Entregas=('grade', 'max'),
#     # O timestamp da última entrega
#     ultima_entrega_timestamp=('delivery_timestamp', 'max')
# ).reset_index()


# # 3b. Encontrar a NOTA da Última Entrega
# # Esta etapa é crucial: encontrar a nota que corresponde ao último timestamp
# df_last_grade = pd.merge(
#     df_limpo,
#     # Faz o merge usando as chaves de agrupamento E o timestamp MÁXIMO
#     df_agregado[['user_id', 'task_question_id', 'ultima_entrega_timestamp']],
#     left_on=['user_id', 'task_question_id', 'delivery_timestamp'], # Merge on delivery_timestamp from df_limpo
#     right_on=['user_id', 'task_question_id', 'ultima_entrega_timestamp'], # Merge on ultima_entrega_timestamp from df_agregado
#     how='inner'
# )

# # Como pode haver múltiplos deliveries no mesmo timestamp máximo, pegamos apenas o primeiro
# df_last_grade = df_last_grade.drop_duplicates(subset=['user_id', 'task_question_id'], keep='first')

# # Extrai a nota e renomeia
# df_last_grade = df_last_grade[['user_id', 'task_question_id', 'grade']].rename(
#     columns={'grade': 'Nota_da_Última_Entrega'}
# )


# # --- 4. Unir e Selecionar Colunas Finais ---
# comparison_df = pd.merge(
#     df_agregado,
#     df_last_grade,
#     on=['user_id', 'task_question_id'],
#     how='left'
# )

# # 5. Selecionar e Renomear as 5 Colunas Solicitadas
# final_df = comparison_df[[
#     'user_id',
#     'task_name',
#     'question_name',
#     'Maior_Nota_das_Entregas',
#     'Nota_da_Última_Entrega'
# ]].rename(columns={
#     'user_id': 'ID do Aluno',
#     'task_name': 'Tarefa',
#     'question_name': 'Questão',
# })


# # --- 6. Salvar e Baixar o Arquivo CSV Final ---
# print(f"\n--- 4. Salvando e baixando '{NOME_ARQUIVO_SAIDA}' ---")
# # Salva o arquivo CSV
# final_df.to_csv(NOME_ARQUIVO_SAIDA, index=False, sep=DELIMITADOR, encoding='utf-8')

# # Permite que o usuário baixe o arquivo gerado (função do Colab)
# files.download(NOME_ARQUIVO_SAIDA)

# print(f"\nProcessamento concluído. O arquivo '{NOME_ARQUIVO_SAIDA}' foi gerado com sucesso!")
# print(f"Total de registros únicos (Aluno x Questão): {len(final_df)}")
# print("\n--- Cabeçalho das Colunas Solicitadas ---")
# display(final_df.head())

# Task
Create a table showing unique user IDs from df_geral, their media_final and faltas, colored based on their presence in df_moodle.

In [ ]:
import pandas as pd

# 1. Get unique user_ids, media_final, faltas, and cod_curso from df_geral
#    We need to drop duplicates first to ensure each user_id has one media_final, faltas, and cod_curso
#    For simplicity, we'll take the first occurrence if a user_id has multiple entries with different grades/faltas/cod_curso
users_geral = df_geral[['user_id', 'media_final', 'faltas', 'cod_curso']].drop_duplicates(subset=['user_id']).copy()

# 2. Filter out rows where 'media_final', 'faltas', or 'cod_curso' are NaN
users_geral_cleaned = users_geral.dropna(subset=['media_final', 'faltas', 'cod_curso']).copy()

# 3. Get unique user_ids from df_moodle
users_moodle = df_moodle['user_id'].unique()

# 4. Create a 'present_in_moodle' column
users_geral_cleaned.loc[:, 'present_in_moodle'] = users_geral_cleaned['user_id'].isin(users_moodle)

# 5. Define a styling function
def color_user_id(row):
    color = 'green' if row['present_in_moodle'] else 'red'
    return [f'color: {color}' if col == 'user_id' else '' for col in row.index]

# 6. Apply the styling and display the table
styled_table = users_geral_cleaned.style.apply(color_user_id, axis=1)

display(styled_table)

# 7. Print the amount of unique IDs after this cleaning
print(f"Number of unique user IDs in df_geral after cleaning 'media_final', 'faltas', and 'cod_curso': {users_geral_cleaned['user_id'].nunique()}")
print(f"Number of unique user IDs in df_moodle: {len(users_moodle)}")

# 8. Generate summary statistics for the markdown cell (based on cleaned data)
total_unique_users_geral = users_geral_cleaned['user_id'].nunique()
users_in_moodle = users_geral_cleaned[users_geral_cleaned['present_in_moodle']]['user_id'].nunique()
users_not_in_moodle = total_unique_users_geral - users_in_moodle

percentage_in_moodle = (users_in_moodle / total_unique_users_geral) * 100
percentage_not_in_moodle = (users_not_in_moodle / total_unique_users_geral) * 100

# Print the findings to update the summary cell
print(f"total_unique_users_geral (cleaned): {total_unique_users_geral}")
print(f"users_in_moodle (cleaned): {users_in_moodle}")
print(f"users_not_in_moodle (cleaned): {users_not_in_moodle}")
print(f"percentage_in_moodle (cleaned): {percentage_in_moodle:.2f}")
print(f"percentage_not_in_moodle (cleaned): {percentage_not_in_moodle:.2f}")

# Task
Create a pandas DataFrame from the summary statistics calculated in the previous step, render this DataFrame as a table using plotly, and save it as a PDF file.

In [ ]:
import pandas as pd
import plotly.graph_objects as go

# Calculate media_final averages
average_media_final_in_moodle = users_geral_cleaned[users_geral_cleaned['present_in_moodle']]['media_final'].mean()
average_media_final_not_in_moodle = users_geral_cleaned[~users_geral_cleaned['present_in_moodle']]['media_final'].mean()

# Prepare the data for the summary table
summary_data = {
    "Metric": [
        "Total Unique Users in df_geral (cleaned)",
        "Users Present in df_moodle",
        "Users Not Present in df_moodle",
        "Percentage Present in df_moodle (%)",
        "Percentage Not Present in df_moodle (%)",
        "Media_final average for students present in Moodle",
        "Media_final average for students not present in Moodle"
    ],
    "Value": [
        total_unique_users_geral,
        users_in_moodle,
        users_not_in_moodle,
        f"{percentage_in_moodle:.2f}",
        f"{percentage_not_in_moodle:.2f}",
        f"{average_media_final_in_moodle:.2f}",
        f"{average_media_final_not_in_moodle:.2f}"
    ]
}

summary_df = pd.DataFrame(summary_data)

# Create a Plotly table
fig = go.Figure(data=[go.Table(
    header=dict(
        values=list(summary_df.columns),
        fill_color='paleturquoise',
        align='left'
    ),
    cells=dict(
        values=[summary_df.Metric, summary_df.Value],
        fill_color='lavender',
        align='left'
    )
)])

fig.update_layout(title_text="Summary Statistics: Moodle Presence and Academic Data")

# Save the table as a PDF
pdf_output_path = "summary_statistics.pdf"
fig.write_image(pdf_output_path, format='pdf')

print(f"Summary table successfully generated and saved as '{pdf_output_path}'")